# Week 73, Day 3: Publishing a package

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w073d3.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
import io, zipfile

buffer = io.BytesIO()
with zipfile.ZipFile(buffer, "w") as z:
    z.writestr("textkit/__init__.py", "def shout(s):\n    return s.upper()\n")
    z.writestr("textkit-0.1.0.dist-info/METADATA", "Metadata-Version: 2.3\nName: textkit\nVersion: 0.1.0\n")

with zipfile.ZipFile(io.BytesIO(buffer.getvalue())) as z:
    print(z.namelist())
    print(z.read("textkit-0.1.0.dist-info/METADATA").decode().splitlines()[1:])

In [ ]:
import base64, hashlib, io, zipfile
from email.parser import Parser

def record_hash(data):
    digest = hashlib.sha256(data).digest()
    return "sha256=" + base64.urlsafe_b64encode(digest).rstrip(b"=").decode()

files = {
    "textkit/__init__.py": b"def shout(s):\n    return s.upper()\n",
    "textkit-0.1.0.dist-info/METADATA": b"Metadata-Version: 2.3\nName: textkit\nVersion: 0.1.0\nRequires-Dist: rich>=13\nRequires-Dist: tomli>=2; python_version < '3.11'\n\nShout things.\n",
    "textkit-0.1.0.dist-info/WHEEL": b"Wheel-Version: 1.0\nGenerator: by-hand\nRoot-Is-Purelib: true\nTag: py3-none-any\n",
}
record = "".join(f"{p},{record_hash(d)},{len(d)}\n" for p, d in files.items())
record += "textkit-0.1.0.dist-info/RECORD,,\n"
print(record)

meta = Parser().parsestr(files["textkit-0.1.0.dist-info/METADATA"].decode())
print(meta["Name"], meta["Version"], meta.get_all("Requires-Dist"))
print(meta.get_payload().strip())

> **Key idea.** An sdist (.tar.gz) is the packed source and must be built to install; a wheel (.whl) is built, and installing it is unzipping. Wheel names are name-version(-build)-python-abi-platform.whl, with the name using underscores; tags can be dot-separated sets, and pip picks the wheel matching its most specific supported tag. A wheel's .dist-info holds METADATA (email headers, Requires-Dist lines, README body), WHEEL, entry_points.txt and RECORD (path, sha256 in unpadded URL-safe base64, size). Release with python -m build, twine check, TestPyPI, then PyPI; versions can't be replaced; use scoped tokens or trusted publishing, never secrets in code.

### Exercise: Reading a wheel's name

Write `parse_wheel(filename)`: return a dict with `"name"`, `"version"`, `"build"` (or `None`), `"python"`, `"abi"` and `"platform"`, where the three tag fields are **sorted lists** (split on `.`). The name must end with `.whl` and have five or six `-`-separated fields; anything else raises `ValueError`. Then `wheel_filename(name, version, tag="py3-none-any")`: the file name for a project, normalising the name (PEP 503) and then replacing `-` with `_`.

In [ ]:
import re

def parse_wheel(filename):
    ___

def wheel_filename(name, version, tag="py3-none-any"):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_names():\n    \"wheel file names\"\n    assert parse_wheel(\"textkit-0.3.0-py3-none-any.whl\") == {\"name\": \"textkit\", \"version\": \"0.3.0\", \"build\": None, \"python\": [\"py3\"], \"abi\": [\"none\"], \"platform\": [\"any\"]}\n    w = parse_wheel(\"six-1.16.0-1-py2.py3-none-any.whl\")\n    assert w[\"build\"] == \"1\" and w[\"python\"] == [\"py2\", \"py3\"]\n    assert parse_wheel(\"numpy-2.1.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl\")[\"platform\"] == [\"manylinux2014_x86_64\", \"manylinux_2_17_x86_64\"]\n    for bad in [\"textkit-0.3.0.tar.gz\", \"textkit-py3-none-any.whl\", \"a-b-c-d-e-f-g.whl\"]:\n        try:\n            parse_wheel(bad)\n            assert False, bad\n        except ValueError:\n            pass\n    assert wheel_filename(\"Text-Kit\", \"1.0\") == \"text_kit-1.0-py3-none-any.whl\"\n    assert wheel_filename(\"zope.interface\", \"6.0\", \"cp312-cp312-win_amd64\") == \"zope_interface-6.0-cp312-cp312-win_amd64.whl\"")

### Exercise: Choosing a wheel

Write `tags(filename)`: the set of every `(python, abi, platform)` triple a wheel supports (all combinations of its compressed tag sets). Then `choose(filenames, supported)`: `supported` is the interpreter's ordered list of triples, most preferred first. Return the wheel whose best (earliest) supported triple comes first; ties go to the wheel listed first. Return `None` if nothing fits.

In [ ]:
import itertools

def tags(filename):
    ___

def choose(filenames, supported):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_choose():\n    \"the most specific compatible wheel\"\n    assert tags(\"six-1.16.0-py2.py3-none-any.whl\") == {(\"py2\", \"none\", \"any\"), (\"py3\", \"none\", \"any\")}\n    supported = [(\"cp312\", \"cp312\", \"manylinux_2_17_x86_64\"), (\"cp312\", \"abi3\", \"manylinux_2_17_x86_64\"),\n                 (\"cp312\", \"none\", \"manylinux_2_17_x86_64\"), (\"py3\", \"none\", \"manylinux_2_17_x86_64\"), (\"py3\", \"none\", \"any\")]\n    files = [\"fast-1.0-py3-none-any.whl\", \"fast-1.0-cp311-abi3-manylinux_2_17_x86_64.whl\",\n             \"fast-1.0-cp312-cp312-manylinux_2_17_x86_64.whl\", \"fast-1.0-cp312-cp312-win_amd64.whl\"]\n    assert choose(files, supported) == \"fast-1.0-cp312-cp312-manylinux_2_17_x86_64.whl\"\n    assert choose(files[:2], supported) == \"fast-1.0-py3-none-any.whl\"\n    assert choose([\"fast-1.0-cp312-cp312-win_amd64.whl\"], supported) is None\n    assert choose([\"a-1-py3-none-any.whl\", \"b-1-py2.py3-none-any.whl\"], supported) == \"a-1-py3-none-any.whl\"")

### Exercise: RECORD and its hashes

Write `record_hash(data)` (bytes to `"sha256=..."`, URL-safe base64 without padding), `make_record(files, record_path)`: the RECORD text for a dict of path to bytes, one `path,hash,size` line per file in the order given, then the line for the RECORD itself with empty hash and size. Then `verify(files, record_text)`: return the sorted list of paths that don't match: files whose hash or size differ, files listed but missing, and files present but not listed (ignoring the RECORD's own entry). Use the `csv` module to read the RECORD, since paths may contain commas (`csv` quotes them).

In [ ]:
import base64, csv, hashlib, io

def record_hash(data):
    ___

def make_record(files, record_path):
    ___

def verify(files, record_text):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_record():\n    \"hashes that installers check\"\n    assert record_hash(b\"\") == \"sha256=47DEQpj8HBSa-_TImW-5JCeuQeRkm5NMpJWZG3hSuFU\"\n    files = {\"pkg/__init__.py\": b\"x = 1\\n\", \"pkg/data,1.txt\": b\"hello\", \"pkg-1.0.dist-info/METADATA\": b\"Name: pkg\\n\"}\n    rec = make_record(files, \"pkg-1.0.dist-info/RECORD\")\n    lines = rec.splitlines()\n    assert lines[0] == \"pkg/__init__.py,\" + record_hash(b\"x = 1\\n\") + \",6\" and lines[1].startswith('\"pkg/data,1.txt\",')\n    assert lines[-1] == \"pkg-1.0.dist-info/RECORD,,\" and len(lines) == 4\n    assert verify(files, rec) == []\n    changed = dict(files, **{\"pkg/__init__.py\": b\"x = 2\\n\"})\n    del changed[\"pkg/data,1.txt\"]\n    changed[\"pkg/extra.py\"] = b\"\"\n    assert verify(changed, rec) == [\"pkg/__init__.py\", \"pkg/data,1.txt\", \"pkg/extra.py\"]")

### Exercise: Reading METADATA

Write `read_metadata(text)` with `email.parser.Parser`: return a dict with `"name"`, `"version"`, `"summary"` (or `None`), `"requires_python"` (or `None`), `"requires"` (a list of every `Requires-Dist` value, in order), `"extras"` (every `Provides-Extra`) and `"description"` (the body after the headers, stripped). Then `runtime_requires(meta)`: the `requires` entries that have no marker mentioning `extra` (the ones installed without any extras).

In [ ]:
from email.parser import Parser

def read_metadata(text):
    ___

def runtime_requires(meta):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_metadata():\n    \"the email-header format\"\n    text = (\"Metadata-Version: 2.3\\nName: textkit\\nVersion: 0.3.0\\nSummary: Small text tools\\n\"\n            \"Requires-Python: >=3.10\\nRequires-Dist: rich>=13\\nRequires-Dist: tomli>=2; python_version < '3.11'\\n\"\n            \"Requires-Dist: pypdf>=4; extra == 'pdf'\\nProvides-Extra: pdf\\n\\n# textkit\\n\\nSmall tools.\\n\")\n    m = read_metadata(text)\n    assert (m[\"name\"], m[\"version\"], m[\"summary\"], m[\"requires_python\"]) == (\"textkit\", \"0.3.0\", \"Small text tools\", \">=3.10\")\n    assert len(m[\"requires\"]) == 3 and m[\"extras\"] == [\"pdf\"] and m[\"description\"] == \"# textkit\\n\\nSmall tools.\"\n    assert runtime_requires(m) == [\"rich>=13\", \"tomli>=2; python_version < '3.11'\"]\n    bare = read_metadata(\"Metadata-Version: 2.3\\nName: x\\nVersion: 1\\n\")\n    assert bare[\"summary\"] is None and bare[\"requires\"] == [] and bare[\"extras\"] == [] and bare[\"description\"] == \"\"")

### Exercise: Building a wheel

Write `build_wheel(name, version, files, requires=(), scripts=None)`: return `(filename, data)`, where `data` is the bytes of a wheel zip containing every package file in `files` (a dict of path to text), plus a `.dist-info` folder named `{normalised name with underscores}-{version}.dist-info` containing `METADATA` (`Metadata-Version: 2.3`, `Name`, `Version`, and a `Requires-Dist` line per requirement), `WHEEL` (`Wheel-Version: 1.0`, `Generator: atlas`, `Root-Is-Purelib: true`, `Tag: py3-none-any`), `entry_points.txt` if `scripts` (a dict of command to `module:function`) is given (a `[console_scripts]` section with `command = target` lines), and finally `RECORD` covering all of them. The file name comes from `wheel_filename` (given).

In [ ]:
import base64, csv, hashlib, io, re, zipfile

def wheel_filename(name, version, tag="py3-none-any"):
    normal = re.sub(r"[-_.]+", "-", name).lower().replace("-", "_")
    return f"{normal}-{version}-{tag}.whl"

def record_hash(data):
    digest = hashlib.sha256(data).digest()
    return "sha256=" + base64.urlsafe_b64encode(digest).rstrip(b"=").decode()

def build_wheel(name, version, files, requires=(), scripts=None):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_build():\n    \"a wheel an installer would accept\"\n    import io, zipfile, csv\n    from email.parser import Parser\n    filename, data = build_wheel(\"Text-Kit\", \"0.3.0\", {\"text_kit/__init__.py\": \"def shout(s):\\n    return s.upper()\\n\"},\n                                 requires=[\"rich>=13\"], scripts={\"tk\": \"text_kit.cli:main\"})\n    assert filename == \"text_kit-0.3.0-py3-none-any.whl\"\n    z = zipfile.ZipFile(io.BytesIO(data))\n    names = z.namelist()\n    info = \"text_kit-0.3.0.dist-info/\"\n    assert set(names) == {\"text_kit/__init__.py\", info + \"METADATA\", info + \"WHEEL\", info + \"entry_points.txt\", info + \"RECORD\"}\n    assert names[-1] == info + \"RECORD\"\n    meta = Parser().parsestr(z.read(info + \"METADATA\").decode())\n    assert meta[\"Name\"] == \"Text-Kit\" and meta[\"Version\"] == \"0.3.0\" and meta.get_all(\"Requires-Dist\") == [\"rich>=13\"]\n    assert \"Tag: py3-none-any\" in z.read(info + \"WHEEL\").decode()\n    assert \"[console_scripts]\" in z.read(info + \"entry_points.txt\").decode() and \"tk = text_kit.cli:main\" in z.read(info + \"entry_points.txt\").decode()\n    rows = list(csv.reader(io.StringIO(z.read(info + \"RECORD\").decode())))\n    assert {r[0] for r in rows} == set(names)\n    for path, h, size in rows:\n        if h:\n            assert h == record_hash(z.read(path)) and int(size) == len(z.read(path)), path\n    _, plain = build_wheel(\"x\", \"1\", {\"x.py\": \"\"})\n    assert \"x-1.dist-info/entry_points.txt\" not in zipfile.ZipFile(io.BytesIO(plain)).namelist()")

### Bug hunt: A hash in the wrong encoding

The RECORD hash is written as a hexadecimal digest, but wheels use **URL-safe base64 without padding**, so an installer would reject the file. Fix `record_hash` so this prints `sha256=LPJNul-wow4m6DsqxbninhsWHlwfp0JecwQzYpOLmCQ True`.

In [ ]:
import base64, hashlib

def record_hash(data):
    return "sha256=" + hashlib.sha256(data).hexdigest()

h = record_hash(b"hello")
print(h, "=" not in h[7:])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"the right encoding\"\n    expect_output(\"sha256=LPJNul-wow4m6DsqxbninhsWHlwfp0JecwQzYpOLmCQ True\")")

### Bug hunt: A hyphen in the file name

The project's name has a hyphen, and it's put into the wheel's file name as it is, so splitting the name on `-` finds too many fields and the tags come out wrong: this prints `python tag: none`. Normalise the name with underscores, so it prints `python tag: py3`.

In [ ]:
def wheel_filename(name, version):
    return f"{name}-{version}-py3-none-any.whl"

filename = wheel_filename("text-kit", "1.0")
fields = filename[:-4].split("-")
print("python tag:", fields[-3] if len(fields) == 5 else fields[-2])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"underscores in names\"\n    expect_output(\"python tag: py3\")")

### Bug hunt: Only the first dependency

`msg["Requires-Dist"]` returns only the **first** of the repeated headers, so the program thinks the project has one dependency and prints `['rich>=13']`. Use the method that returns every value, so it prints `['rich>=13', 'regex']`.

In [ ]:
from email.parser import Parser

text = "Metadata-Version: 2.3\nName: textkit\nVersion: 1.0\nRequires-Dist: rich>=13\nRequires-Dist: regex\n"
msg = Parser().parsestr(text)
print([msg["Requires-Dist"]])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"repeated headers\"\n    expect_output(\"['rich>=13', 'regex']\")")

### Puzzle: A safe release

Arrange the steps of a release in order: install the tools, build, check, try it on the practice index, then publish. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the platform tag of a pure-Python wheel
q = ___   # how many (python, abi, platform) triples "py2.py3-none-any" covers
r = ___   # True or False: installing a wheel runs code from the project
s = ___   # the file in .dist-info that lists every file with its hash
t = ___   # True or False: a version uploaded to PyPI can be replaced by uploading it again

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == \"any\" and q == 2 and r is False, \"p, q and r\"\n    assert s == \"RECORD\" and t is False, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: installing a wheel

Write `install(wheel_bytes, site)`: install a wheel into `site`, a dict standing in for the environment's files (path to bytes). Steps: open the zip; find the single `*.dist-info/RECORD`; **refuse** (raise `ValueError`) if any entry's path is absolute (starts with `/`) or contains a `..` part, if any file in the zip isn't listed in RECORD, or if any listed hash or size doesn't match (use `record_hash`, given). Then copy every file into `site` under the same path; add `INSTALLER` (containing `b"atlas\n"`) to the `.dist-info` folder; and for each console script in `entry_points.txt` (read with `configparser`), add `bin/COMMAND` containing the wrapper text from yesterday (`import sys`, `from MODULE import FIRST`, `sys.exit(FUNCTION())`, each on its own line, ending with a newline), encoded. Return the sorted list of every path added to `site`.

In [ ]:
import base64, configparser, csv, hashlib, io, zipfile

def record_hash(data):
    digest = hashlib.sha256(data).digest()
    return "sha256=" + base64.urlsafe_b64encode(digest).rstrip(b"=").decode()

def install(wheel_bytes, site):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def make(files, tamper=None, extra=None):\n    import io, zipfile, csv\n    info = \"tk-1.0.dist-info/\"\n    contents = {p: d.encode() for p, d in files.items()}\n    out = io.StringIO()\n    w = csv.writer(out, lineterminator=\"\\n\")\n    for p, d in contents.items():\n        w.writerow([p, record_hash(d), len(d)])\n    w.writerow([info + \"RECORD\", \"\", \"\"])\n    contents[info + \"RECORD\"] = out.getvalue().encode()\n    if tamper:\n        contents[tamper] = b\"changed\"\n    if extra:\n        contents[extra] = b\"surprise\"\n    buf = io.BytesIO()\n    with zipfile.ZipFile(buf, \"w\") as z:\n        for p, d in contents.items():\n            z.writestr(p, d)\n    return buf.getvalue()\n\nFILES = {\"tk/__init__.py\": \"X = 1\\n\", \"tk/cli.py\": \"def main():\\n    return 0\\n\",\n         \"tk-1.0.dist-info/METADATA\": \"Metadata-Version: 2.3\\nName: tk\\nVersion: 1.0\\n\",\n         \"tk-1.0.dist-info/entry_points.txt\": \"[console_scripts]\\ntk = tk.cli:main\\ntk-admin = tk.cli:Admin.run\\n\"}\n\ndef test_install():\n    \"a checked install\"\n    site = {}\n    added = install(make(FILES), site)\n    assert added == [\"bin/tk\", \"bin/tk-admin\", \"tk-1.0.dist-info/INSTALLER\", \"tk-1.0.dist-info/METADATA\",\n                     \"tk-1.0.dist-info/RECORD\", \"tk-1.0.dist-info/entry_points.txt\", \"tk/__init__.py\", \"tk/cli.py\"]\n    assert site[\"tk/cli.py\"] == b\"def main():\\n    return 0\\n\" and site[\"tk-1.0.dist-info/INSTALLER\"] == b\"atlas\\n\"\n    assert site[\"bin/tk\"] == b\"import sys\\nfrom tk.cli import main\\nsys.exit(main())\\n\"\n    assert site[\"bin/tk-admin\"] == b\"import sys\\nfrom tk.cli import Admin\\nsys.exit(Admin.run())\\n\"\n\ndef test_refusals():\n    \"tampered, unlisted and unsafe files\"\n    for wheel in (make(FILES, tamper=\"tk/cli.py\"), make(FILES, extra=\"tk/sneaky.py\"),\n                  make(dict(FILES, **{\"../outside.py\": \"x\"})), make(dict(FILES, **{\"/etc/x\": \"x\"}))):\n        site = {}\n        try:\n            install(wheel, site)\n            assert False\n        except ValueError:\n            assert site == {}, \"nothing is installed when the wheel is refused\"")

## Warm-up

That's very nearly a wheel: a zip file with the package's files at the top and a `.dist-info` folder describing it. By the end of today you'll have built one that pip would accept.

## Two kinds of distribution

Building a project (with the backend from yesterday's `[build-system]`) produces up to two **distributions**:

- A **source distribution** (**sdist**), `textkit-0.3.0.tar.gz`: the source tree with `pyproject.toml`, packed up. Installing one means **building** it on the user's machine, which needs the build backend and, for extension modules, a C compiler. It's the archival form of a release, and the fallback when no wheel fits.
- A **wheel**, `textkit-0.3.0-py3-none-any.whl`: a **built** distribution, ready to install. Installing is just unzipping files into the right places, with no code from the project run at all. For pure-Python projects one wheel fits every platform; projects with compiled code (Week 68) publish one wheel per Python version and platform.

`pip install textkit` prefers a compatible wheel, and falls back to the sdist. So publish both: the wheel for speed, the sdist for completeness.

## Wheel names and tags

A wheel's file name says exactly what it's for (the format comes from PEPs 427 and 425):

```text
{name}-{version}(-{build})?-{python tag}-{abi tag}-{platform tag}.whl
textkit-0.3.0-py3-none-any.whl
numpy-2.1.0-cp312-cp312-manylinux_2_17_x86_64.whl
```

- **name** is normalised with **underscores** instead of hyphens (`text_kit`), because hyphens separate the fields. **build** is an optional number, used rarely, to re-release the same version.
- The **python tag** says which interpreter: `py3` (any Python 3), `cp312` (CPython 3.12 only).
- The **ABI tag** says which compiled interface: `none` (no compiled code), `cp312`, or `abi3` (the stable ABI, Week 68 Day 5).
- The **platform tag**: `any`, or something like `win_amd64`, `macosx_11_0_arm64` or `manylinux_2_17_x86_64` (Linux with glibc 2.17 or newer).
- Any of the three tags may be a **compressed set** joined by dots: `py2.py3-none-any` fits Python 2 and 3.

pip builds an ordered list of the tag triples the running interpreter supports, most specific first, and picks the wheel matching the **earliest** one. `pip debug --verbose` prints that list.

## Inside a wheel

A wheel unpacks to the package files plus one folder, `{name}-{version}.dist-info`, holding:

- **`METADATA`**: the project's metadata in email-header format (`Name:`, `Version:`, `Summary:`, a `Requires-Dist:` line per dependency, `Requires-Python:`, `Provides-Extra:` ...), with the README as the body after a blank line. Python's `email.parser` reads it.
- **`WHEEL`**: about the wheel itself (`Wheel-Version: 1.0`, `Generator:`, `Root-Is-Purelib: true`, `Tag:` lines).
- **`entry_points.txt`** (if any): an INI file with a `[console_scripts]` section, from which the installer writes the command wrappers (yesterday's `wrapper`).
- **`RECORD`**: a CSV line for **every file** in the wheel: `path,sha256=HASH,size`. The hash is the SHA-256 digest encoded in **URL-safe base64 without `=` padding**. RECORD's own line has empty hash and size. Installers check every file against it, and record installed files in it so `pip uninstall` knows what to remove.

The **sdist** is a `.tar.gz` of the source with a `PKG-INFO` file (the same format as `METADATA`) at the top.

## Releasing

The usual release, on your own computer:

```bash
python -m pip install --upgrade build twine
```

```bash
python -m build
```

```bash
python -m twine check dist/*
```

```bash
python -m twine upload --repository testpypi dist/*
```

1. **`python -m build`** runs the backend in an isolated environment and writes the sdist and wheel to `dist/`. (`uv build` does the same.)
2. **`twine check`** validates the metadata, including that the README will display on PyPI.
3. Upload to **TestPyPI** first, a separate practice index, and install from it into a fresh environment to see what users will get.
4. Upload to **PyPI**. A version, once uploaded, can **never** be replaced, even if you delete it: fix mistakes with a new version (Day 4).

**Credentials.** Never put a password or token in code, `pyproject.toml` or a repository. PyPI uses **API tokens** (scoped to one project where possible), or better, **trusted publishing**: you tell PyPI which CI workflow (Week 74) may publish the project, and the CI system proves its identity to PyPI with a short-lived token, so there's no long-lived secret to leak at all.

**Before the first release**: check the name is free on PyPI (normalised), choose a licence and include its file, write a README that says what the project does and how to install it, and fill in `requires-python`, the classifiers and the project URLs.

## Practice

## Quiz

## Daily challenge

## Before you go

Download a wheel of a small package you use (`pip download --no-deps rich`, on your own computer), open it with `python -m zipfile -l`, and read its `METADATA` and `RECORD`. Tomorrow: **versioning and changelogs**: what version numbers promise, how to choose the next one, and how to tell users what changed.